# AI Orchestrator — Colab Smoke Test

FastAPI 서버를 띄우지 않고 `pipeline.run()`을 **파이썬 함수로 직접 호출**해서
4가지 라우팅 경로를 그대로 검증하는 노트북이야.

**흐름:**
1. 레포 clone + 의존성 설치
2. HuggingFace 모델 ID 변수 설정 (router / summary / answer)
3. 각 모델 로드 & `LocalModelRegistry.register()`
4. `pipeline.run()`으로 시나리오 테스트
   - DIRECT_ANSWER
   - RETRIEVE_DOC (PDF 업로드)
   - SEARCH_PREP_THEN_RETRIEVE
   - ASK_CLARIFICATION


## 1. 레포 clone & 의존성 설치

In [1]:
# 본인 레포 주소로 바꾸세요
REPO_URL = "https://github.com/2026-Pretrained-Conversational-Model/ai-engine.git"
REPO_DIR = "/content/ai-orchestrator"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("repo already cloned")


Cloning into '/content/ai-orchestrator'...
remote: Enumerating objects: 245, done.
remote: Counting objects: 100% (245/245), done.
remote: Compressing objects: 100% (180/180), done.
remote: Total 245 (delta 118), reused 188 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (245/245), 151.98 KiB | 11.69 MiB/s, done.
Resolving deltas: 100% (118/118), done.


In [2]:
# 핵심 의존성 (이미 requirements.txt 있지만 Colab 기본 이미지 기준으로 추가 설치)
!pip install -q "fastapi==0.115.0" "pydantic==2.9.2" "pydantic-settings==2.5.2" \
    "pypdf==5.0.0" "sentence-transformers==3.1.1" "faiss-cpu==1.8.0" \
    "transformers>=4.45" "accelerate>=0.34" "bitsandbytes>=0.43" "nest_asyncio" "boto3"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.4/149.4 kB 15.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.9/434.9 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.8/292.8 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.3/245.3 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 141.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. 환경변수 + sys.path 설정 **(반드시 app import 전에)**

In [3]:
import os, sys

# Python 오케스트레이터가 local 백엔드를 쓰도록 전환
os.environ["LLM_BACKEND"] = "local"

# Colab GPU에 bge-m3(1024d) 올리기엔 부담되니 소형 임베딩으로 교체
os.environ["EMBEDDING_MODEL_NAME"] = "jhgan/ko-sroberta-multitask"
os.environ["EMBEDDING_DIM"] = "768"
os.environ["EMBEDDING_DEVICE"] = "cuda"

# 업로드 파일 저장 경로 (Colab 임시)
os.environ["LOCAL_FILE_DIR"] = "/content/_orchestrator_files"

# 세션 캡 여유있게
os.environ["SESSION_MAX_BYTES"] = str(20 * 1024 * 1024)

# 레포를 import path에 추가
REPO_DIR = "/content/ai-orchestrator"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("env ready")


env ready


## 3. HuggingFace 모델 ID 설정

세 가지 role에 쓸 모델을 HF Hub ID로 지정해. 같은 모델을 여러 role에 공유해도 되고,
예산이 빠듯하면 `ROUTER_MODEL_ID`/`SUMMARY_MODEL_ID`를 `None`으로 두면
해당 role은 **휴리스틱/no-op 폴백**으로 동작해 (answer 모델만 있으면 파이프라인은 돈다).

In [4]:
# 필요에 맞게 변경
ANSWER_MODEL_ID  = "Qwen/Qwen2.5-7B-Instruct"
ROUTER_MODEL_ID  = None   # None 두면 휴리스틱 폴백
SUMMARY_MODEL_ID = None   # None 두면 summary no-op
MEMORY_MODEL_ID  = "g34634/qwen2.5-3b-memory-summary-v1"  # Memory State Generator

# 4bit 양자화 사용 (T4 16GB 기준 7B+3B+1.5B 동시 로드 가능)
USE_4BIT = True


## 4. 모델 로드 & 등록

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def _load(model_id: str, tokenizer_id: str | None = None):
    if model_id is None:
        return None, None

    tok_id = tokenizer_id or model_id

    print(f"loading model={model_id}")
    print(f"loading tokenizer={tok_id}")

    tok = AutoTokenizer.from_pretrained(tok_id, trust_remote_code=True)

    # Qwen 계열 안정화
    tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    kwargs = dict(device_map="auto", trust_remote_code=True)

    if USE_4BIT:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
    else:
        kwargs["torch_dtype"] = torch.float16

    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)

    model.config.pad_token_id = tok.pad_token_id
    model.config.eos_token_id = tok.eos_token_id
    if hasattr(model, "generation_config") and model.generation_config is not None:
        model.generation_config.pad_token_id = tok.pad_token_id
        model.generation_config.eos_token_id = tok.eos_token_id

    model.eval()
    print(f"  -> ready on {next(model.parameters()).device}")
    return model, tok

BASE_QWEN_TOKENIZER_ID = "Qwen/Qwen2.5-3B-Instruct"

answer_model, answer_tok = _load(ANSWER_MODEL_ID)
router_model, router_tok = _load(ROUTER_MODEL_ID)
summary_model, summary_tok = _load(SUMMARY_MODEL_ID)

# Memory State Generator 모델 로드
memory_model, memory_tok = _load(MEMORY_MODEL_ID, tokenizer_id=BASE_QWEN_TOKENIZER_ID)


loading model=Qwen/Qwen2.5-7B-Instruct
loading tokenizer=Qwen/Qwen2.5-7B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/6.17G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  -> ready on cuda:0


In [6]:
ls

ai-orchestrator/  sample_data/


In [7]:
cd ai-orchestrator/

/content/ai-orchestrator


In [8]:
ls

app/                   Dockerfile  README.md         run.sh
ARCHITECTURE_NOTES.md  notebooks/  requirements.txt


In [9]:
!pip install boto3

In [10]:
!find /content/ai-orchestrator/app -maxdepth 4 -type f

/content/ai-orchestrator/app/services/orchestrator/response_finalizer.py
/content/ai-orchestrator/app/services/orchestrator/augmentation.py
/content/ai-orchestrator/app/services/orchestrator/search_prep.py
/content/ai-orchestrator/app/services/orchestrator/__init__.py
/content/ai-orchestrator/app/services/orchestrator/pipeline.py
/content/ai-orchestrator/app/services/session/session_getter.py
/content/ai-orchestrator/app/services/session/session_cleaner.py
/content/ai-orchestrator/app/services/session/session_manager.py
/content/ai-orchestrator/app/services/session/__init__.py
/content/ai-orchestrator/app/services/session/session_updater.py
/content/ai-orchestrator/app/services/session/session_creator.py
/content/ai-orchestrator/app/services/session/memory_monitor.py
/content/ai-orchestrator/app/services/pdf/pdf_saver.py
/content/ai-orchestrator/app/services/pdf/document_cache.py
/content/ai-orchestrator/app/services/pdf/pdf_ingest.py
/content/ai-orchestrator/app/services/pdf/pdf_index

In [11]:
import sys, os

REPO_DIR = "/content/ai-orchestrator"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("sys.path[0]:", sys.path[0])

print("\n--- services tree ---")
os.system("find /content/ai-orchestrator/app/services -maxdepth 4 -type f")

print("\n--- local_registry search ---")
os.system("find /content/ai-orchestrator -type f | grep local_registry")

print("\n--- LocalModelRegistry search ---")
os.system("grep -R 'class LocalModelRegistry' /content/ai-orchestrator")

sys.path[0]: /content/ai-orchestrator

--- services tree ---

--- local_registry search ---

--- LocalModelRegistry search ---


0

In [12]:
from app.services.llm.local_registry import LocalModelRegistry

LocalModelRegistry.clear()  # 노트북 재실행 안전

if answer_model is not None:
    LocalModelRegistry.register("answer", answer_model, answer_tok,
                                 device="cuda", max_new_tokens=512)

if router_model is not None:
    LocalModelRegistry.register("router", router_model, router_tok,
                                 device="cuda", max_new_tokens=24)

if summary_model is not None:
    LocalModelRegistry.register("summary", summary_model, summary_tok,
                                 device="cuda", max_new_tokens=400)

# Memory State Generator를 memory role로 등록
if memory_model is not None:
    LocalModelRegistry.register("memory", memory_model, memory_tok,
                                 device="cuda", max_new_tokens=256)

print("registered roles:", LocalModelRegistry.list_roles())


2026-04-14 13:44:24,722 [INFO] app.services.llm.local_registry: Registered local model: role=answer device=cuda
INFO:app.services.llm.local_registry:Registered local model: role=answer device=cuda
2026-04-14 13:44:24,723 [INFO] app.services.llm.local_registry: Registered local model: role=memory device=cuda
INFO:app.services.llm.local_registry:Registered local model: role=memory device=cuda


registered roles: ['answer', 'memory']


## 5. `pipeline.run()` 호출 헬퍼

Colab의 Jupyter 이벤트 루프 위에서 async 함수를 돌리려면 `nest_asyncio`가 필요해.

In [13]:
!pip uninstall -y faiss faiss-cpu faiss-gpu numpy
!pip install -q "numpy<2" "faiss-cpu==1.7.4"

Found existing installation: faiss-cpu 1.8.0
Uninstalling faiss-cpu-1.8.0:
  Successfully uninstalled faiss-cpu-1.8.0
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.5 MB/s eta 0:00:00
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement faiss-cpu==1.7.4 (from versions: 1.8.0, 1.8.0.post1, 1.9.0, 1.9.0.post1, 1.10.0, 1.11.0, 1.11.0.post1, 1.12.0, 1.13.0, 1.13.1, 1.13.2)
ERROR: No matching distribution found for faiss-cpu==1.7.4


In [14]:
!pip install langchain_text_splitters

In [15]:
import asyncio, nest_asyncio, uuid, json
nest_asyncio.apply()

# app 모듈 import (이 시점부터 config가 load됨 — 그래서 os.environ 설정을 먼저 해야 함)
from app.schemas.request import ChatRequest
from app.services.orchestrator.pipeline import run as pipeline_run
from app.services.session.session_manager import SessionManager
from app.services.embedding.embedding_singleton import EmbeddingSingleton

# 임베딩 모델 워밍업 (원래 FastAPI lifespan에서 하는 일)
EmbeddingSingleton.warmup()

def chat(session_id: str, user_text: str, file_path: str | None = None,
         file_name: str | None = None) -> dict:
    req = ChatRequest(
        session_id=session_id,
        user_text=user_text,
        file_path=file_path,
        file_name=file_name,
    )
    resp = asyncio.get_event_loop().run_until_complete(pipeline_run(req))
    return resp.model_dump()

def dump_session(session_id: str):
    s = asyncio.get_event_loop().run_until_complete(
        SessionManager.instance().get(session_id)
    )
    if s is None:
        print("(session purged or not found)")
        return
    print(json.dumps(s.model_dump(mode="json"), ensure_ascii=False, indent=2))

print("helper ready")


2026-04-14 13:44:40,098 [INFO] app.services.embedding.embedding_singleton: Loading embedding model: jhgan/ko-sroberta-multitask
INFO:app.services.embedding.embedding_singleton:Loading embedding model: jhgan/ko-sroberta-multitask


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-04-14 13:44:48,408 [INFO] app.services.embedding.embedding_singleton: Embedding model ready
INFO:app.services.embedding.embedding_singleton:Embedding model ready


helper ready


## 시나리오 A — `DIRECT_ANSWER`

PDF 없음, 평범한 질문. 라우터가 DIRECT_ANSWER를 반환하고 RAG는 안 탐.

In [16]:
import inspect
from app.services.orchestrator.pipeline import run
print(inspect.getsource(run))

async def run(req: ChatRequest) -> ChatResponse:
    log_stage_start(
        logger,
        "TURN",
        session_id=req.session_id,
        user_text=req.user_text,
        has_pdf=bool(req.file_path and req.file_name),
        has_image=bool(req.image_b64),
        file_name=req.file_name,
    )

    # 1) session load ----------------------------------------------------------
    log_stage_start(logger, "SESSION_LOAD", session_id=req.session_id)
    session = await get_or_create(req.session_id)
    log_stage_end(logger, "SESSION_LOAD", session_id=req.session_id)
    log_memory(logger, session, "after_session_load")

    # 2) optional PDF attach + background ingest start -------------------------
    ingest_task = None
    if req.file_path and req.file_name:
        log_stage_start(
            logger,
            "PDF_ATTACH_OR_INGEST",
            session_id=req.session_id,
            file_name=req.file_name,
            file_path=req.file_path,
        )

        current_name 

In [17]:
sid_a = f"test-a-{uuid.uuid4().hex[:6]}"

r1 = chat(sid_a, "안녕, 내 이름은 김예슬이야.")
print("=== turn 1 ===")
print("answer_type:", r1["answer_type"])
print("answer     :", r1["answer"][:500])
print()

r2 = chat(sid_a, "내 이름이 뭐라고?")
print("=== turn 2 (후속) ===")
print("answer_type:", r2["answer_type"])
print("answer     :", r2["answer"][:500])


2026-04-14 13:44:48,430 [INFO] app.services.orchestrator.pipeline: [START] TURN | {"session_id": "test-a-babb46", "user_text": "안녕, 내 이름은 김예슬이야.", "has_pdf": false, "has_image": false, "file_name": null}
INFO:app.services.orchestrator.pipeline:[START] TURN | {"session_id": "test-a-babb46", "user_text": "안녕, 내 이름은 김예슬이야.", "has_pdf": false, "has_image": false, "file_name": null}
2026-04-14 13:44:48,431 [INFO] app.services.orchestrator.pipeline: [START] SESSION_LOAD | {"session_id": "test-a-babb46"}
INFO:app.services.orchestrator.pipeline:[START] SESSION_LOAD | {"session_id": "test-a-babb46"}
2026-04-14 13:44:48,433 [INFO] app.services.session.session_manager: SessionManager initialized
INFO:app.services.session.session_manager:SessionManager initialized
2026-04-14 13:44:48,435 [INFO] app.services.orchestrator.pipeline: [END] SESSION_LOAD | {"session_id": "test-a-babb46"}
INFO:app.services.orchestrator.pipeline:[END] SESSION_LOAD | {"session_id": "test-a-babb46"}
2026-04-14 13:44:48,436 

=== turn 1 ===
answer_type: AnswerType.MULTITURN_ONLY
answer     : 안녕하세요 김예슬님! 만나서 반갑습니다. 무엇을 도와드릴까요?



2026-04-14 13:44:53,169 [INFO] app.services.orchestrator.pipeline: [END] LLM_GENERATE | {"answer_length": 44, "answer_preview": "김예슬이라고 말씀하셨어요. 처음에 말씀하셨듯이. 무엇 다른 도움이 필요하신가요?", "answer_type": "AnswerType.MULTITURN_ONLY"}
INFO:app.services.orchestrator.pipeline:[END] LLM_GENERATE | {"answer_length": 44, "answer_preview": "김예슬이라고 말씀하셨어요. 처음에 말씀하셨듯이. 무엇 다른 도움이 필요하신가요?", "answer_type": "AnswerType.MULTITURN_ONLY"}
2026-04-14 13:44:53,170 [INFO] app.services.orchestrator.pipeline: [START] FINALIZE | {}
INFO:app.services.orchestrator.pipeline:[START] FINALIZE | {}
2026-04-14 13:44:53,171 [INFO] app.services.orchestrator.pipeline: [MEMORY] {"label": "before_finalize", "memory": "None", "pdf": {"active_pdf": null, "doc_summary": {"one_line": "", "section_summaries": []}, "pdf_index": {"chunks": []}, "ingest_status": "idle", "ingest_error": ""}, "runtime": {"active_resource_type": "none", "active_pdf_id": null, "last_answer_type": "multiturn_only", "pdf_ingest_status": "idle", "last_router_decis

=== turn 2 (후속) ===
answer_type: AnswerType.MULTITURN_ONLY
answer     : 김예슬이라고 말씀하셨어요. 처음에 말씀하셨듯이. 무엇 다른 도움이 필요하신가요?


## 시나리오 B — `RETRIEVE_DOC`

PDF 업로드 + 문서 키워드 포함 질문. 라우터가 RETRIEVE_DOC을 반환하고
ingest 완료를 기다린 뒤 FAISS retrieve → RAG 프롬프트로 answer LLM 호출.

In [18]:
# 샘플 PDF 업로드 (또는 본인 PDF 경로 사용)
from google.colab import files
uploaded = files.upload()   # 파일 선택 창
pdf_name = list(uploaded.keys())[0]
pdf_path = f"/content/{pdf_name}"
with open(pdf_path, "wb") as f:
    f.write(uploaded[pdf_name])
print("saved to", pdf_path)


Saving [공유용]평가계획서_AINLP_4회차.pdf to [공유용]평가계획서_AINLP_4회차.pdf
saved to /content/[공유용]평가계획서_AINLP_4회차.pdf


In [19]:
from app.core.config import settings
print("LLM_BACKEND =", settings.LLM_BACKEND)

LLM_BACKEND = local


In [ ]:
sid_b = f"test-b-{uuid.uuid4().hex[:6]}"

r1 = chat(sid_b, "이 문서가 뭐에 대한 거야? 간단히 요약해줘.",
          file_path=pdf_path, file_name=pdf_name)
print("=== turn 1 ===")
print("answer_type:", r1["answer_type"])
print("answer     :", r1["answer"][:800])
print()

r2 = chat(sid_b, "이 문서에서 수상이력 표로 정리해줘.")
print("=== turn 2 ===")
print("answer_type:", r2["answer_type"])
print("answer     :", r2["answer"][:800])


2026-04-14 13:48:23,808 [INFO] app.services.orchestrator.pipeline: [START] TURN | {"session_id": "test-b-99ed63", "user_text": "이 문서가 뭐에 대한 거야? 간단히 요약해줘.", "has_pdf": true, "has_image": false, "file_name": "[공유용]평가계획서_AINLP_4회차.pdf"}
INFO:app.services.orchestrator.pipeline:[START] TURN | {"session_id": "test-b-99ed63", "user_text": "이 문서가 뭐에 대한 거야? 간단히 요약해줘.", "has_pdf": true, "has_image": false, "file_name": "[공유용]평가계획서_AINLP_4회차.pdf"}
2026-04-14 13:48:23,809 [INFO] app.services.orchestrator.pipeline: [START] SESSION_LOAD | {"session_id": "test-b-99ed63"}
INFO:app.services.orchestrator.pipeline:[START] SESSION_LOAD | {"session_id": "test-b-99ed63"}
2026-04-14 13:48:23,810 [INFO] app.services.orchestrator.pipeline: [END] SESSION_LOAD | {"session_id": "test-b-99ed63"}
INFO:app.services.orchestrator.pipeline:[END] SESSION_LOAD | {"session_id": "test-b-99ed63"}
2026-04-14 13:48:23,811 [INFO] app.services.orchestrator.pipeline: [MEMORY] {"label": "after_session_

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-14 13:48:25,219 [INFO] app.services.embedding.faiss_store: FAISS cache saved: 6e07a9d5
INFO:app.services.embedding.faiss_store:FAISS cache saved: 6e07a9d5
2026-04-14 13:48:25,221 [INFO] app.services.pdf.pdf_ingest: PDF ingest ready: session=test-b-99ed63 file_id=pdf_01a43565
INFO:app.services.pdf.pdf_ingest:PDF ingest ready: session=test-b-99ed63 file_id=pdf_01a43565
2026-04-14 13:48:25,223 [INFO] app.services.orchestrator.pipeline: [MEMORY] {"label": "after_pdf_ready", "memory": "None", "pdf": {"active_pdf": {"file_id": "pdf_01a43565", "file_name": "[공유용]평가계획서_AINLP_4회차.pdf", "file_path": "/content/_orchestrator_files/test-b-99ed63/[공유용]평가계획서_AINLP_4회차.pdf", "uploaded_at": "2026-04-14 13:48:23.817101", "file_size": 245213, "page_count": 15, "mime_type": "application/pdf", "file_hash": "6e07a9d579a52750545d636339f073946af8b68eb3d818a1d01896e5acb9eada"}, "doc_summary": {"one_line": "작성일 : 2026년 03월 27일\n심 화_기 업  연 계 형  실 전 \n프 로 젝 트  중 심 \nA I과 정_자 연 

=== turn 1 ===
answer_type: AnswerType.MULTITURN_WITH_PDF
answer     : 이 문서는 AI와 자연어 처리(NLP) 관련 교육 프로그램의 평가 방법과 내용에 대해 설명하고 있습니다. 주요 내용은 다음과 같습니다:

1. **평가 대상자 자료 관리**: 사전평가에서 지원자들이 제출한 자기소개서, Python 시험 결과, 기존 KDT 과정 수료 결과물을 검토하고, 2차 평가에서는 면접을 통해 지원자의 프로그래밍 능력, CS 및 머신러닝 이해, 교육 참여 의지 등을 평가합니다.

2. **평가 일정**: 교과목별 평가 일정이 정해져 있으며, 프로젝트 평가를 통해 학습 성장률을 측정합니다.

3. **평가 방법**: 교과목별로 지필 평가, 객관식/주관식 퀴즈, 실습 과제 등을 통해 평가하며, 평가 결과가 하위 10%인 경우 주강사와의 상담을 통해 원인을 파악하고 조치를 취합니다.

4. **프로젝트 평가**: 프로젝트 평가는 교육 전과 후의 학습 성장률을 측정하고, 피드백을 제공하여 학습 성장을 높입니다.

5. **교육 프로그램 개선**: 평가 결과를 바탕으로 교육 프로그램을 개선하고, 훈련생 만족도를 높이기 위한 다양한 노력이 이루어집니다.

이 문서는 AI와 자연어 처리 분야의 교육 프로그램에서의 평가 체계와 방법에 대해 자세히 설명하고 있습니다.



2026-04-14 13:49:26,912 [INFO] app.services.orchestrator.pipeline: [END] LLM_GENERATE | {"answer_length": 963, "answer_preview": "다음은 문서에서 제공된 정보를 바탕으로 수상 이력 표를 정리한 것입니다:\n\n| 순서 | 항목          | 내용                                                                 |\n|------|--------------|---------------------------------------------", "answer_type": "AnswerType.MULTITURN_WITH_PDF"}
INFO:app.services.orchestrator.pipeline:[END] LLM_GENERATE | {"answer_length": 963, "answer_preview": "다음은 문서에서 제공된 정보를 바탕으로 수상 이력 표를 정리한 것입니다:\n\n| 순서 | 항목          | 내용                                                                 |\n|------|--------------|---------------------------------------------", "answer_type": "AnswerType.MULTITURN_WITH_PDF"}
2026-04-14 13:49:26,914 [INFO] app.services.orchestrator.pipeline: [START] FINALIZE | {}
INFO:app.services.orchestrator.pipeline:[START] FINALIZE | {}
2026-04-14 13:49:26,915 [INFO] app.services.orchestrator.pipeline: [MEMORY] {"label": "before_finalize", "me

=== turn 2 ===
answer_type: AnswerType.MULTITURN_WITH_PDF
answer     : 다음은 문서에서 제공된 정보를 바탕으로 수상 이력 표를 정리한 것입니다:

| 순서 | 항목          | 내용                                                                 |
|------|--------------|----------------------------------------------------------------------|
| 1    | 평가 대상자 자료 관리 | 사전평가에서 지원자들이 제출한 자기소개서, Python 시험 결과, 기 KDT 과정 수료 결과물을 검토하고, 2차 평가에서는 면접을 통해 지원자의 프로그래밍 능력, CS 및 머신러닝 이해, 교육 참여 의지 등을 평가합니다. |
| 2    | 평가 일정      | - 자연어 처리 개론 01: 26.03.27 (지필 평가) <br> - 자연어 처리 개론 02: 26.04.02 (지필 평가) <br> - 프로젝트 01: 26.04.10 (프로젝트 산출물 평가) <br> - 프로젝트 02: 26.04.17 (프로젝트 산출물 평가) <br> - 종합 프로젝트: 26.05.28 (프로젝트 산출물 평가) |
| 3    | 평가 방법      | - 지필 평가 <br> - 객관식/주관식 퀴즈 <br> - 실습 과제 <br> - 프로젝트 평가 (교육 전과 후의 학습 성장률 측정) <br> - 피드백 제공 (강사님 및 보조강사, 참여기업 담당자 피드백) |
| 4    | 평가 결과 활용 | - 평가 결과를 분석하여 훈련생들의 성취도 평가 <br> - 차기 회차 프로그램 개발 및 


## 시나리오 C — `SEARCH_PREP_THEN_RETRIEVE`

PDF가 세션에 붙어있는 상태에서 참조 표현(`"그거"`, `"그 표"`)로 질문.
라우터가 SEARCH_PREP_THEN_RETRIEVE를 반환하고 search_prep으로 쿼리 재작성 후 retrieve.

In [ ]:
# 시나리오 B에서 쓴 session을 그대로 이어서 질문
r3 = chat(sid_b, "만들어준 표 내용 요역해줘.")
print("=== turn 3 (참조 표현) ===")
print("answer_type :", r3["answer_type"])
print("answer      :", r3["answer"][:800])

# 실제 라우터가 어느 label로 판정했는지 세션에서 확인
dump_session(sid_b)


2026-04-14 13:49:26,927 [INFO] app.services.orchestrator.pipeline: [START] TURN | {"session_id": "test-b-99ed63", "user_text": "만들어준 표 내용 요역해줘.", "has_pdf": false, "has_image": false, "file_name": null}
INFO:app.services.orchestrator.pipeline:[START] TURN | {"session_id": "test-b-99ed63", "user_text": "만들어준 표 내용 요역해줘.", "has_pdf": false, "has_image": false, "file_name": null}
2026-04-14 13:49:26,929 [INFO] app.services.orchestrator.pipeline: [START] SESSION_LOAD | {"session_id": "test-b-99ed63"}
INFO:app.services.orchestrator.pipeline:[START] SESSION_LOAD | {"session_id": "test-b-99ed63"}
2026-04-14 13:49:26,930 [INFO] app.services.orchestrator.pipeline: [END] SESSION_LOAD | {"session_id": "test-b-99ed63"}
INFO:app.services.orchestrator.pipeline:[END] SESSION_LOAD | {"session_id": "test-b-99ed63"}
2026-04-14 13:49:26,931 [INFO] app.services.orchestrator.pipeline: [MEMORY] {"label": "after_session_load", "memory": "None", "pdf": {"active_pdf": {"file_id": "pdf_01a43565", "file_name": "[

=== turn 3 (참조 표현) ===
answer_type : AnswerType.MULTITURN_WITH_PDF
answer      : ### 수상 이력 표 요약

| 순서 | 항목          | 내용                                                                 |
|------|--------------|----------------------------------------------------------------------|
| 1    | 평가 대상자 자료 관리 | - 사전평가1차 평가: 자기소개서, Python 시험 결과, 기 KDT 과정 수료 결과물 제출<br>- 2차 평가: 면접, 프로그래밍 능력, CS 및 머신러닝 이해, 교육 참여 의지, 소통 능력 검토 |
| 2    | 평가 일정      | - 자연어 처리 개론 01: 26.03.27 (지필 평가)<br>- 자연어 처리 개론 02: 26.04.02 (지필 평가)<br>- 프로젝트 01: 26.04.10 (프로젝트 산출물 평가)<br>- 프로젝트 02: 26.04.17 (프로젝트 산출물 평가)<br>- 종합 프로젝트: 26.05.28 (프로젝트 산출물 평가) |
| 3    | 평가 방법      | - 지필 평가<br>- 객관식/주관식 퀴즈<br>- 실습 과제<br>- 프로젝트 평가 (교육 전과 후의 학습 성장률 측정)<br>- 피드백 제공 (강사님 및 보조강사, 참여기업 담당자 피드백) |
| 4    | 평가 결과 활용 | - 평가 결과를 분석하여 훈련생들의 성취도 평가<br>- 차기 회차 프로그램 개발 및 운영 시 난이도 조절 및 교육 수준 조절<br>- 훈련생 만족도 향상을 위한 노력 (고충 및 건의사항 
{
  "session_meta": {
    "session_id": "test-b-99ed63",
    "created_at": "2026-04-14T13:48:23.810439",
    "last_acc

## 시나리오 D — `ASK_CLARIFICATION`

빈 세션 + PDF 없음 + 지시어만 있는 짧은 질문. 라우터가 ASK_CLARIFICATION으로 판정하고
**LLM 호출 자체를 스킵**해서 고정 재질문 메시지를 반환.

In [ ]:
sid_d = f"test-d-{uuid.uuid4().hex[:6]}"

r = chat(sid_d, "내 이름이 뭐라고?")
print("answer_type:", r["answer_type"])
print("answer     :", r["answer"])
# LLM 호출이 없었으므로 GPU 부하 없이 즉시 반환되어야 함


2026-04-14 13:50:00,494 [INFO] app.services.orchestrator.pipeline: [START] TURN | {"session_id": "test-d-368301", "user_text": "내 이름이 뭐라고?", "has_pdf": false, "has_image": false, "file_name": null}
INFO:app.services.orchestrator.pipeline:[START] TURN | {"session_id": "test-d-368301", "user_text": "내 이름이 뭐라고?", "has_pdf": false, "has_image": false, "file_name": null}
2026-04-14 13:50:00,495 [INFO] app.services.orchestrator.pipeline: [START] SESSION_LOAD | {"session_id": "test-d-368301"}
INFO:app.services.orchestrator.pipeline:[START] SESSION_LOAD | {"session_id": "test-d-368301"}
2026-04-14 13:50:00,496 [INFO] app.services.orchestrator.pipeline: [END] SESSION_LOAD | {"session_id": "test-d-368301"}
INFO:app.services.orchestrator.pipeline:[END] SESSION_LOAD | {"session_id": "test-d-368301"}
2026-04-14 13:50:00,497 [INFO] app.services.orchestrator.pipeline: [MEMORY] {"label": "after_session_load", "memory": "None", "pdf": {"active_pdf": null, "doc_summary": {"one_line": "", "section_summar

answer_type: AnswerType.MULTITURN_ONLY
answer     : 당신의 이름을 물어보시는 것看起来您是想询问对方的名字。请问您是要我转述对方的名字，还是有其他问题需要帮助解答呢？如果您是在问别人的名字，请提供更多的上下文信息。


In [ ]:
sid_e = f"test-e-{uuid.uuid4().hex[:6]}"

r1 = chat(sid_e, "내 이름은 김예슬이야.")
print("=== turn 1 ===")
print("answer_type:", r1["answer_type"])
print("answer     :", r1["answer"][:800])
print()

r2 = chat(sid_e, "내 이름이 뭐라고?")
print("=== turn 2 ===")
print("answer_type:", r2["answer_type"])
print("answer     :", r2["answer"][:800])


## 세션 상태 덤프 (디버깅용)

Memory State Generator가 세션에 무엇을 남겼는지, router 결정이 무엇이었는지,
PDF ingest 상태가 어떤지 전부 JSON으로 볼 수 있어.

In [ ]:
dump_session(sid_a)
print("---")
dump_session(sid_b)


## 세션 정리

테스트 끝나면 메모리/파일/FAISS 모두 purge. (`SESSION_MAX_BYTES` 초과 시에도 자동으로 이게 일어남.)

In [ ]:
from app.services.session.session_cleaner import purge_session

for sid in [sid_a, sid_b, sid_d]:
    asyncio.get_event_loop().run_until_complete(purge_session(sid, reason="test_done"))
print("purged")


In [ ]:
from app.services.llm.local_registry import LocalModelRegistry
print(LocalModelRegistry.has('summary'))

# 직접 테스트
result = LocalModelRegistry.generate('summary', '테스트 입력')
print(result)

In [ ]:
s = asyncio.get_event_loop().run_until_complete(
    SessionManager.instance().get('test-a-bf2d08')
)
print('narrative:', s.conversation.summary.narrative)